# Stream Clustering: Online Algorithms

## Learning Objectives

1. **Define** the online clustering problem for data streams
2. **Describe** the GRGPF algorithm for streaming cluster maintenance
3. **Implement** a simplified online clustering with cluster summaries
4. **Discuss** the CluStream framework for time-evolving clusters

## The Online Clustering Problem

**Batch clustering:** all $n$ points available; standard k-means or hierarchical.

**Stream clustering:** points arrive one at a time; we must:
- Update clusters incrementally with $O(1)$ or $O(k)$ work per point
- Answer "what is the current clustering?" at any time
- Use only $O(k)$ or $O(k \log n)$ space

**Challenge:** we can't re-run k-means each time; we need an incremental summary.

## Cluster Sufficient Statistics

For a cluster $C$ of $|C|$ points in $\mathbb{R}^d$, maintain:

$$N = |C|, \quad \text{SUM} = \sum_{x \in C} x, \quad \text{SUMSQ} = \sum_{x \in C} x^2$$

From these we can derive:
- Centroid: $\mu = \text{SUM} / N$
- Variance: $\sigma^2 = \text{SUMSQ}/N - (\text{SUM}/N)^2$
- Radius: $\text{RMSE} = \sqrt{\sum_d \sigma_d^2 / d}$

These statistics can be merged: $SS(A \cup B) = SS(A) + SS(B)$ (component-wise).
This is the core of BFR and all streaming cluster algorithms.

In [ ]:
import math, random

class ClusterSummary:
    """Sufficient statistics for a cluster."""
    def __init__(self, d):
        self.N = 0
        self.SUM = [0.0]*d
        self.SUMSQ = [0.0]*d
        self.d = d

    def add(self, point):
        self.N += 1
        for i,x in enumerate(point):
            self.SUM[i] += x; self.SUMSQ[i] += x**2

    def centroid(self):
        return [self.SUM[i]/self.N for i in range(self.d)] if self.N > 0 else [0]*self.d

    def variance(self):
        if self.N == 0: return [0]*self.d
        return [self.SUMSQ[i]/self.N - (self.SUM[i]/self.N)**2 for i in range(self.d)]

    def radius(self):
        return math.sqrt(sum(self.variance())/self.d) if self.d > 0 else 0.0

    def merge(self, other):
        merged = ClusterSummary(self.d)
        merged.N = self.N + other.N
        merged.SUM = [a+b for a,b in zip(self.SUM, other.SUM)]
        merged.SUMSQ = [a+b for a,b in zip(self.SUMSQ, other.SUMSQ)]
        return merged

def dist(x, y):
    return math.sqrt(sum((a-b)**2 for a,b in zip(x,y)))

class OnlineKMeans:
    """Simple online k-means using cluster summaries."""
    def __init__(self, k, d, seed=0):
        self.k = k; self.d = d
        self.clusters = []  # List of ClusterSummary
        self.initialized = False

    def update(self, point):
        if len(self.clusters) < self.k:
            cs = ClusterSummary(self.d)
            cs.add(point)
            self.clusters.append(cs)
        else:
            # Assign to nearest centroid
            dists = [dist(point, cs.centroid()) for cs in self.clusters]
            nearest = min(range(self.k), key=lambda i: dists[i])
            self.clusters[nearest].add(point)

    def get_centroids(self):
        return [cs.centroid() for cs in self.clusters]

    def inertia(self, points):
        cs = self.get_centroids()
        return sum(min(dist(p,c)**2 for c in cs) for p in points)

# Simulate a stream
rng = random.Random(42)
k, d = 3, 2
true_centers = [(0,0),(10,0),(5,8)]
stream = []
for cx,cy in true_centers:
    for _ in range(200):
        stream.append([cx+rng.gauss(0,1.5), cy+rng.gauss(0,1.5)])
rng.shuffle(stream)

okm = OnlineKMeans(k, d)
snapshots = []
for t, point in enumerate(stream):
    okm.update(point)
    if t in (50, 150, 299, 499):
        inertia = okm.inertia(stream[:t+1])
        snapshots.append((t+1, inertia, [f"({c[0]:.1f},{c[1]:.1f})" for c in okm.get_centroids()]))

print(f"{'t':>5} {'Inertia':>10}  Centroids")
for t, inertia, centers in snapshots:
    print(f"{t:>5} {inertia:>10.0f}  {centers}")

## CluStream Framework

CluStream (Aggarwal et al. 2003) separates clustering into two phases:

**Online micro-clustering:**
- Maintain $q$ micro-clusters as running summaries (N, SUM, SUMSQ, time info)
- Each new point: assign to nearest micro-cluster if within threshold; else create new micro-cluster (delete oldest if at capacity)
- Snapshots saved at exponentially spaced time points (for time window queries)

**Offline macro-clustering:**
- Apply k-means to the $q$ micro-cluster centroids (weighted by $N$)
- Answer "what is the clustering over time window $[t_1, t_2]$?"

**Advantages:**
- Handles concept drift (recent data gets more micro-clusters)
- $O(q)$ storage; very fast per-point update
- Can answer historical queries via stored snapshots